In [67]:
%pip install statsmodels


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/martanarozhnyak/py312/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [68]:
# Importing libraries

import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats

In [69]:
# Prepare data for Z-test

# Step 1: Load the csv file of Q2: Has the category mix shifted over time?

df_1 = pd.read_csv('/Users/martanarozhnyak/Desktop/iowa_liquor_sales/02_categories_shift.csv')

In [70]:
# Step 2: Preview the dataset

df_1.head()

,category_name,bottles_2018,bottles_2024,share_2018,share_2024,share_change_pp
0,WHISKEY LIQUEUR,1470029,4607050.0,5.79,14.66,8.87
1,100% AGAVE TEQUILA,437565,1190675.0,1.72,3.79,2.07
2,AMERICAN SCHNAPPS,575400,1062207.0,2.26,3.38,1.12
3,STRAIGHT BOURBON WHISKIES,1157952,1731985.0,4.56,5.51,0.95
4,TEMPORARY & SPECIALTY PACKAGES,241017,568115.0,0.95,1.81,0.86


In [71]:
# Step 3: filter to the WHISKEY LIQUEUR row
whiskey_row = df_1[df_1['category_name'] == 'WHISKEY LIQUEUR']

# Step 4: extract the scalar values for the whiskey liqueur bottles in 2018 and 2024
# Use .iloc[0] to extract the raw scalar — without it, filtering returns
# a pandas Series (with index and metadata), which would break the z-test.
whiskey_liqueur_bottles_2018 = whiskey_row['bottles_2018'].iloc[0]
whiskey_liqueur_bottles_2024 = whiskey_row['bottles_2024'].iloc[0]

In [72]:
# Step 5: Getting the total number of all liquor bottles in 2018 and 2024
total_bottles_2018 = df['bottles_2018'].sum()
total_bottles_2024 = df['bottles_2024'].sum()

In [73]:
# Step 6: Checking the outputs

print(f"Whiskey Liqueur 2018: {whiskey_liqueur_bottles_2018:,.0f}")
print(f"Whiskey Liqueur 2024: {whiskey_liqueur_bottles_2024:,.0f}")
print(f"Total 2018: {total_bottles_2018:,.0f}")
print(f"Total 2024: {total_bottles_2024:,.0f}")
print(f"Share 2018: {whiskey_liqueur_bottles_2018 / total_bottles_2018 * 100:.2f}%")
print(f"Share 2024: {whiskey_liqueur_bottles_2024 / total_bottles_2024 * 100:.2f}%")

Whiskey Liqueur 2018: 1,470,029
Whiskey Liqueur 2024: 4,607,050
Total 2018: 25,409,905
Total 2024: 31,422,333
Share 2018: 5.79%
Share 2024: 14.66%


In [74]:
# Step 7: Set up inputs for the Z-test

count = [whiskey_liqueur_bottles_2024, whiskey_liqueur_bottles_2018]
nobs = [total_bottles_2024, total_bottles_2018]

In [75]:
# Step 8: Run the Z-test

z_stat, p_value = proportions_ztest(count, nobs)

print(f"Z-statistic: {z_stat:,.2f}")
print(f"P-value: {p_value:.2e}")

Z-statistic: 1,076.64
P-value: 0.00e+00


In [76]:
# Step 9: Summary of Z-test

# Statistical test: Two-proportion z-test on WHISKEY LIQUEUR's volume share between 2018 and 2024

# H₀: share_2018 = share_2024
# H₁: share_2018 ≠ share_2024 (two-sided)
# α = 0.05

# Result: z = 1,076.64, p-value = 0.0

# Conclusion: Since p < 0.05, we reject H₀ and conclude that the change in WHISKEY LIQUEUR's volume
# share between 2018 and 2024 is statistically significant: the +8.87 percentage-point shift
# cannot be attributed to random variation

In [77]:
# Prepare data for T-test

# Step 1: Load the csv file of Q4: Are there meaningful price tiers, and do categories price differently?

df_2 = pd.read_csv('/Users/martanarozhnyak/Desktop/iowa_liquor_sales/04_category_price_tiers.csv')

In [78]:
# Step 2: Preview the dataset
df_2.head()

,category_name,transactions,mean_price,stddev_price,p25,median_price,p75
0,SINGLE MALT SCOTCH,16244,67.59,87.40,41.97,52.47,71.24
1,SINGLE BARREL BOURBON WHISKIES,7674,44.86,22.78,28.50,37.50,51.00
2,BOTTLED IN BOND BOURBON,7685,35.53,18.61,20.25,37.50,45.00
3,TEMPORARY & SPECIALTY PACKAGES,74009,44.18,60.85,24.74,31.26,56.25
4,MEZCAL,2809,33.89,15.89,23.25,30.00,38.25


In [82]:
# Step 3: Create the necessary variables for the selected two categories for T-test

# SINGLE MALT SCOTCH
mean_a = 67.59
std_a = 87.40
n_a = 16244

# AMERICAN VODKAS
mean_b = 10.64
std_b = 6.78
n_b = 403838

In [83]:
# Step 4: Run the T-test
t_stat, p_val = stats.ttest_ind_from_stats(
    mean1=mean_a, std1=std_a, nobs1=n_a,
    mean2=mean_b, std2=std_b, nobs2=n_b,
    equal_var=False  # Welch's t-test, safer when variances differ
)

print(f"T-statistic: {t_stat:,.2f}")
print(f"P-value: {p_val:.2e}")

T-statistic: 83.04
P-value: 0.00e+00


In [84]:
# Step 5: Summary of T-test

# Statistical test: Welch's two-sample t-test on per-bottle wholesale price (state_bottle_retail), 
# Single Malt Scotch vs American Vodkas, 2024 transactions only

# H₀: mean_price(Scotch) = mean_price(Vodka)
# H₁: mean_price(Scotch) ≠ mean_price(Vodka) (two-sided)
# α = 0.05

# Result: t = 83.04, p-value = 0.00e+00
# Conclusion: Since p < 0.05, we reject H₀ and conclude that the mean wholesale bottle price differs significantly between
# premium and value categories